## Avance 2: Proceso de Extracción y Transformación (Yelp API)

##### 2.1 Configuración del Entorno e Importación

In [1]:
import pandas as pd
import requests
import numpy as np
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:

#url de la API
api_url='https://api.yelp.com/v3/businesses/search'

#estos datos corresponden a una cuenta de usuario creada previamente
clientid='GWOCZh9-BmZxtdsAjr7Gug'
apikey='FHVvXoNmTXIl9DuxYis7AV5uLPujm9MLwrhgs5NgvCfaOxd3V6mxt6dQU8eEqYJiGxe816XATx7ufWjbMWqbV-2Uku1jxBJv8BGRC74NroLPl27PDQqs0tDixit-YHYx'
headers={'Authorization':'Bearer %s'%apikey}

In [4]:
ciudad = "Miami" # Se puede elegir cualquier ciudad del dataset entregado

#### 2.2 Extracción de Datos vía API (Request)

In [5]:
# De esta manera se consulta el API de Yelp para obtener información sobre restaurantes en la ciudad especificada
params={'term':'restaurants','location': ciudad,'limit':50}
response=requests.get(api_url,params=params,headers=headers)
data=response.json()
data

{'businesses': [{'id': 'K3ukx2e11xTRtYBU01dmrA',
   'alias': 'salty-flame-miami',
   'name': 'Salty Flame',
   'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/_6wShTvzfucB5dxFZv4Mpg/o.jpg',
   'is_closed': False,
   'url': 'https://www.yelp.com/biz/salty-flame-miami?adjust_creative=GWOCZh9-BmZxtdsAjr7Gug&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=GWOCZh9-BmZxtdsAjr7Gug',
   'review_count': 258,
   'categories': [{'alias': 'asianfusion', 'title': 'Asian Fusion'},
    {'alias': 'steak', 'title': 'Steakhouses'},
    {'alias': 'cocktailbars', 'title': 'Cocktail Bars'}],
   'rating': 4.3,
   'coordinates': {'latitude': 25.76022, 'longitude': -80.19267},
   'transactions': ['delivery', 'pickup'],
   'location': {'address1': '1414 Brickell Ave',
    'address2': None,
    'address3': '',
    'city': 'Miami',
    'zip_code': '33131',
    'country': 'US',
    'state': 'FL',
    'display_address': ['1414 Brickell Ave', 'Miami, FL 33131']},
   'phone': '+1305563897

In [6]:
#Veamos los keys del diccionario recibido
data.keys()

dict_keys(['businesses', 'total', 'region'])

In [7]:
#El primer elemento del diccionario indica el total de restaurants existentes en la API
print('En total la base de datos registra %d restaurants'%data['total'])

En total la base de datos registra 5800 restaurants


In [8]:
data['businesses'] # Hasta acá se le entregaría a los estudiantes

[{'id': 'K3ukx2e11xTRtYBU01dmrA',
  'alias': 'salty-flame-miami',
  'name': 'Salty Flame',
  'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/_6wShTvzfucB5dxFZv4Mpg/o.jpg',
  'is_closed': False,
  'url': 'https://www.yelp.com/biz/salty-flame-miami?adjust_creative=GWOCZh9-BmZxtdsAjr7Gug&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=GWOCZh9-BmZxtdsAjr7Gug',
  'review_count': 258,
  'categories': [{'alias': 'asianfusion', 'title': 'Asian Fusion'},
   {'alias': 'steak', 'title': 'Steakhouses'},
   {'alias': 'cocktailbars', 'title': 'Cocktail Bars'}],
  'rating': 4.3,
  'coordinates': {'latitude': 25.76022, 'longitude': -80.19267},
  'transactions': ['delivery', 'pickup'],
  'location': {'address1': '1414 Brickell Ave',
   'address2': None,
   'address3': '',
   'city': 'Miami',
   'zip_code': '33131',
   'country': 'US',
   'state': 'FL',
   'display_address': ['1414 Brickell Ave', 'Miami, FL 33131']},
  'phone': '+13055638972',
  'display_phone': '(305) 563-89

### 2.3 Extracción y transformación de la información de la pagina

In [9]:
#Transformamos la información recibida en un dataframe de pandas
contador = 0
lista_consolidada = []

while contador < 200:
    params={'term':'restaurants','location': ciudad,'limit':50,'offset':contador}
    response=requests.get(api_url,params=params,headers=headers)

    data=response.json()
    
    df_yelp = pd.json_normalize(
        data["businesses"],
        record_path="categories",
        meta=[
            "id","name","review_count","rating","price",
            ["coordinates","latitude"],
            ["coordinates","longitude"],
            ["location","address1"]
        ],
    sep='_',
    errors='ignore'
    )
    lista_consolidada.append(df_yelp)
    contador += 50

In [10]:
# Informacion general del dataframe original df_yelp
print("\n--- 1.2 Información general del Dataframe ---")
display(df_yelp.info())


--- 1.2 Información general del Dataframe ---
<class 'pandas.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   alias                  107 non-null    str   
 1   title                  107 non-null    str   
 2   id                     107 non-null    str   
 3   name                   107 non-null    str   
 4   review_count           107 non-null    object
 5   rating                 107 non-null    object
 6   price                  74 non-null     str   
 7   coordinates_latitude   107 non-null    object
 8   coordinates_longitude  107 non-null    object
 9   location_address1      104 non-null    str   
dtypes: object(4), str(6)
memory usage: 8.5+ KB


None

In [11]:
# Cantidad de restaurantes obtenidos en cada uno de los 4 limites del bucle (cada uno con 50 restaurantes hasta 200).
for i, df in enumerate(lista_consolidada):
    print(f"DataFrame {i+1}: {len(df)} restaurantes")
    

DataFrame 1: 101 restaurantes
DataFrame 2: 112 restaurantes
DataFrame 3: 106 restaurantes
DataFrame 4: 107 restaurantes


#### 2.3.1 Integración de Fuentes y Consolidación del Dataset Final

In [12]:
# Unir todos los dataframes obtenidos en uno solo e informacion general del dataframe consolidado df_yelp_consolidado
df_yelp_miami = pd.concat(lista_consolidada, ignore_index=True)
print("\n--- 1.4 Información general del Dataframe consolidado ---")
display(df_yelp_miami.info())


--- 1.4 Información general del Dataframe consolidado ---
<class 'pandas.DataFrame'>
RangeIndex: 426 entries, 0 to 425
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   alias                  426 non-null    str   
 1   title                  426 non-null    str   
 2   id                     426 non-null    str   
 3   name                   426 non-null    str   
 4   review_count           426 non-null    object
 5   rating                 426 non-null    object
 6   price                  277 non-null    str   
 7   coordinates_latitude   426 non-null    object
 8   coordinates_longitude  426 non-null    object
 9   location_address1      423 non-null    str   
dtypes: object(4), str(6)
memory usage: 33.4+ KB


None

##### 2.3.2 Exploración del Dataframe consolidado

In [13]:
# Cantidad de columnas y filas en cada dataframe
print("DataFrame original:")
print(df_yelp.shape)
print("DataFrame consolidado:")
print(df_yelp_miami.shape)

DataFrame original:
(107, 10)
DataFrame consolidado:
(426, 10)


In [14]:
# ver dataframe consolidado
df_yelp_miami.head(10)

,alias,title,id,name,review_count,rating,price,coordinates_latitude,coordinates_longitude,location_address1
0,asianfusion,Asian Fusion,K3ukx2e11xTRtYBU01dmrA,Salty Flame,258,4.3,NaN,25.76022,-80.19267,1414 Brickell Ave
1,steak,Steakhouses,K3ukx2e11xTRtYBU01dmrA,Salty Flame,258,4.3,NaN,25.76022,-80.19267,1414 Brickell Ave
2,cocktailbars,Cocktail Bars,K3ukx2e11xTRtYBU01dmrA,Salty Flame,258,4.3,NaN,25.76022,-80.19267,1414 Brickell Ave
3,cuban,Cuban,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,3192,4.4,$$,25.765607,-80.219218,1442 SW 8th St
4,bars,Bars,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,3192,4.4,$$,25.765607,-80.219218,1442 SW 8th St
5,venues,Venues & Event Spaces,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,3192,4.4,$$,25.765607,-80.219218,1442 SW 8th St
6,italian,Italian,oxtMfBGmVNE18pFVuw7lFg,Fratellino,1901,4.8,$$$,25.74917,-80.26011,264 Miracle Mile
7,sushi,Sushi Bars,O4gh_gZPp1zW9QtC1aMHWw,House of Food Porn,178,4.8,NaN,25.832632,-80.200107,197 NW 62nd St
8,asianfusion,Asian Fusion,O4gh_gZPp1zW9QtC1aMHWw,House of Food Porn,178,4.8,NaN,25.832632,-80.200107,197 NW 62nd St
9,wine_bars,Wine Bars,O4gh_gZPp1zW9QtC1aMHWw,House of Food Porn,178,4.8,NaN,25.832632,-80.200107,197 NW 62nd St


In [15]:
# porcentaje de valores nulos por columna
porcentaje_nulos = df_yelp_miami.isnull().mean() * 100
print("\n--- 1.4 Porcentaje de valores nulos por columna ---")
print(porcentaje_nulos)


--- 1.4 Porcentaje de valores nulos por columna ---
alias                     0.000000
title                     0.000000
id                        0.000000
name                      0.000000
review_count              0.000000
rating                    0.000000
price                    34.976526
coordinates_latitude      0.000000
coordinates_longitude     0.000000
location_address1         0.704225
dtype: float64


In [16]:
filas_con_errores = df_yelp_miami[df_yelp_miami.isnull().any(axis=1)]
print(f"Cantidad de filas con errores: {len(filas_con_errores)}")

Cantidad de filas con errores: 149


In [17]:
print(df_yelp_miami.isnull().sum())

alias                      0
title                      0
id                         0
name                       0
review_count               0
rating                     0
price                    149
coordinates_latitude       0
coordinates_longitude      0
location_address1          3
dtype: int64


#### 2.3.3 Limpieza y transformación columna: "precio"

In [18]:
# Agrupar por columna 'price' y contar la cantidad de filas
agrupado_price = df_yelp_miami.groupby('price').size().reset_index(name='cantidad')
print(agrupado_price)

  price  cantidad
0     $        16
1    $$       189
2   $$$        61
3  $$$$        11


In [19]:
# Reemplazar los valores nulos de la columna 'price' por sin información
price_v2 = "Sin Información"
df_yelp_miami['price'] = df_yelp_miami['price'].fillna(price_v2)

#Agrupar por columna 'price' y contar la cantidad de filas
agrupado_price = df_yelp_miami.groupby('price').size().reset_index(name='cantidad')
print(agrupado_price)

             price  cantidad
0                $        16
1               $$       189
2              $$$        61
3             $$$$        11
4  Sin Información       149


In [20]:
# Homologar elementos de la columna 'price' por "bajo", "medio", "alto" y "muy alto"
df_yelp_miami['price'] = df_yelp_miami['price'].replace({
    '$': 'Bajo',
    '$$': 'Medio',
    '$$$': 'Alto',
    '$$$$': 'Muy Alto'
})
#Agrupar por columna 'price' y contar la cantidad de filas
agrupado_price = df_yelp_miami.groupby('price').size().reset_index(name='cantidad')
print(agrupado_price)

             price  cantidad
0             Alto        61
1             Bajo        16
2            Medio       189
3         Muy Alto        11
4  Sin Información       149


#### 2.3.4 Limpieza de la columna 'location_address1'

In [21]:
# Imputar valor "desconocido" a los valores nulos de la columna 'location_address1'
df_yelp_miami['location_address1'] = df_yelp_miami['location_address1'].fillna('Sin Información')
print("\n--- 1.5 Porcentaje de valores nulos por columna después de la imputación ---")
print(df_yelp_miami.isnull().mean() * 100)


--- 1.5 Porcentaje de valores nulos por columna después de la imputación ---
alias                    0.0
title                    0.0
id                       0.0
name                     0.0
review_count             0.0
rating                   0.0
price                    0.0
coordinates_latitude     0.0
coordinates_longitude    0.0
location_address1        0.0
dtype: float64


### 2.4 Dataframe despues de la limpieza

In [22]:
# Ver la cantidad de resultados antes y despues de convertir la información de la API:
print(f"DataFrame original: {len(df_yelp)} restaurantes")
print(f"DataFrame consolidado: {len(df_yelp_miami)} restaurantes")

DataFrame original: 107 restaurantes
DataFrame consolidado: 426 restaurantes


In [23]:
# Ver como quedó el dataframe consolidado
display(df_yelp_miami.head(10))

,alias,title,id,name,review_count,rating,price,coordinates_latitude,coordinates_longitude,location_address1
0,asianfusion,Asian Fusion,K3ukx2e11xTRtYBU01dmrA,Salty Flame,258,4.3,Sin Información,25.76022,-80.19267,1414 Brickell Ave
1,steak,Steakhouses,K3ukx2e11xTRtYBU01dmrA,Salty Flame,258,4.3,Sin Información,25.76022,-80.19267,1414 Brickell Ave
2,cocktailbars,Cocktail Bars,K3ukx2e11xTRtYBU01dmrA,Salty Flame,258,4.3,Sin Información,25.76022,-80.19267,1414 Brickell Ave
3,cuban,Cuban,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,3192,4.4,Medio,25.765607,-80.219218,1442 SW 8th St
4,bars,Bars,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,3192,4.4,Medio,25.765607,-80.219218,1442 SW 8th St
5,venues,Venues & Event Spaces,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,3192,4.4,Medio,25.765607,-80.219218,1442 SW 8th St
6,italian,Italian,oxtMfBGmVNE18pFVuw7lFg,Fratellino,1901,4.8,Alto,25.74917,-80.26011,264 Miracle Mile
7,sushi,Sushi Bars,O4gh_gZPp1zW9QtC1aMHWw,House of Food Porn,178,4.8,Sin Información,25.832632,-80.200107,197 NW 62nd St
8,asianfusion,Asian Fusion,O4gh_gZPp1zW9QtC1aMHWw,House of Food Porn,178,4.8,Sin Información,25.832632,-80.200107,197 NW 62nd St
9,wine_bars,Wine Bars,O4gh_gZPp1zW9QtC1aMHWw,House of Food Porn,178,4.8,Sin Información,25.832632,-80.200107,197 NW 62nd St


### 2.5 Exportación archivo a csv

In [24]:
# exportar csv
df_yelp_miami.to_csv('df_yelp_miami.csv', index=False)